## Ingest Fact Data from Bronze to Silver Layer


In [0]:
# Import Required Libraries
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

In [0]:
%run /Workspace/Users/yklynk@gmail.com/Azure_databricks_data_engineering_project_shopvista_ecomm/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema,gold_schema)

In [0]:
# Widget
dbutils.widgets.text('catalog', 'shopvista', 'catalog')
dbutils.widgets.text('storage_account_name', 'storageshopvista', 'storage_account_name')
dbutils.widgets.text('container_name', 'shopvista-raw-data', 'container_namee')

In [0]:
# Retrieve widget values for catalog and storage account details
catalog = dbutils.widgets.get('catalog')
storage_account_name = dbutils.widgets.get('storage_account_name')
container_name = dbutils.widgets.get('container_name')
print(catalog, storage_account_name, container_name)

## ORDER ITEMS

In [0]:
# Streaming read from bronze: order_items
df_bronze= spark.readStream\
.format('delta')\
.option("checkpointLocation", "/Volumes/shopvista/raw/checkpoints/bronze")\
.table(f'{catalog}.{bronze_schema}.order_items')

%md
- Perform Transformations and Cleaning

In [0]:
# change coupon code from null value to 'No coupon'
df_silver = df_bronze.fillna('No Coupon', subset = ["coupon_code"])
#display(df_silver.limit(10))

In [0]:
# Transformation: Convert 'Two' → 2 and cast to Integer
df_silver = df_silver.withColumn(
    "quantity",
    F.when(F.col("quantity") == "Two", 2).otherwise(F.col("quantity")).cast("int")
)

In [0]:
# Transformation : Remove any '$' or other symbols from unit_price, keep only numeric
df_silver = df_silver.withColumn(
    "unit_price",
    F.regexp_replace("unit_price", "[$]", "").cast("double")
)

In [0]:
# Transformation : Remove '%' from discount_pct and cast to double
df_silver = df_silver.withColumn(
    "discount_pct",
    F.regexp_replace("discount_pct", "%", "").cast("double")
)

#display(df_silver.limit(10))

In [0]:
# Transformation : coupon code processing (convert to lower)
df_silver = df_silver.withColumn(
    "coupon_code", F.lower(F.trim(F.col("coupon_code")))
)

In [0]:
# Transformation : channel processing 
df_silver = df_silver.withColumn(
    "channel",
    F.when(F.col("channel") == "web", "Website")
    .when(F.col("channel") == "app", "Mobile")
    .otherwise(F.col("channel")),
)

#display(df_silver.limit(5))

In [0]:
#Transformation : Add processed time 
df_silver = df_silver.withColumn(
    "processed_time", F.current_timestamp()
)

#display(df_silver.limit(5))

In [0]:
# Transformation : coupon code processing (convert to lower)
df_silver = df_silver.withColumn(
    "coupon_code", F.lower(F.trim(F.col("coupon_code")))
)
#display(df_silver.limit(30))

## Save to Silver Table

In [0]:
# silver checkpoint location
silver_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/silver/fact_order_items"
print(silver_checkpoint_path)

In [0]:
# Upsert function: merge order_items microbatch into silver table
def upsert_to_silver(microBatchDF, batchId):
    table_name = f"{catalog}.silver.order_items"
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
        spark.sql(
            f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    else:
        deltaTable = DeltaTable.forName(spark, table_name)
        deltaTable.alias("silver_table").merge(
            microBatchDF.alias("batch_table"),
            "silver_table.order_id = batch_table.order_id AND silver_table.item_seq = batch_table.item_seq",
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()    

    



In [0]:
# This line is running a Structured Streaming job that:
# - Reads incremental data from Bronze (df).
# - For each batch → applies upsert_to_silver (update if exists, insert if not).
# - Writes into a Silver Delta table with schema evolution enabled.
# - Uses checkpointing for recovery.
# - Runs in batch-like mode (once or availableNow), not continuous streaming.

df_silver.writeStream.trigger(availableNow=True).foreachBatch(
    upsert_to_silver
).format("delta").option("checkpointLocation", silver_checkpoint_path).option(
    "mergeSchema", "true"
).outputMode(
    "update"
).trigger(
    once=True
).start().awaitTermination()

## ORDER RETURNS

In [0]:
# Streaming read from bronze: order_returns
df_bronze= spark.readStream\
.format('delta')\
.option("checkpointLocation", "/Volumes/shopvista/raw/checkpoints/bronze")\
.table(f'{catalog}.{bronze_schema}.order_returns')

%md
- Perform Transformations and Cleaning

In [0]:
# change order_dt to datetime format
df_silver = df_bronze.withColumn(
    'order_dt',
    F.to_timestamp(F.col('order_dt'), 'yyyy-MM-dd')
)
#display(df_silver.limit(5))

In [0]:
#Convert `reason` column to uppercase and trim whitespace.
df_silver = df_silver.withColumn(
    'reason',
    F.upper(F.trim(F.col('reason')))
)
#display(df_silver.limit(5))

In [0]:
# Add processed_time column
df_silver = df_silver.withColumn(
    'processed_time',
    F.current_timestamp()
)
#display(df_silver.limit(5))

## Save to Silver Table

In [0]:
# silver checkpoint location
silver_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/silver/fact_order_returns"
print(silver_checkpoint_path)

In [0]:
# Upsert function: merge order_returns microbatch into silver table
def upsert_to_silver(microBatchDF, batchId):
    table_name = f"{catalog}.silver.order_returns"
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
        spark.sql(
            f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    else:
        deltaTable = DeltaTable.forName(spark, table_name)
        deltaTable.alias("silver_table").merge(
            microBatchDF.alias("batch_table"),
            "silver_table.order_returns = batch_table.order_returns AND silver_table.item_seq = batch_table.item_seq",
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()    

    



In [0]:
# This line is running a Structured Streaming job that:
# - Reads incremental data from Bronze (df).
# - For each batch → applies upsert_to_silver (update if exists, insert if not).
# - Writes into a Silver Delta table with schema evolution enabled.
# - Uses checkpointing for recovery.
# - Runs in batch-like mode (once or availableNow), not continuous streaming.

df_silver.writeStream.trigger(availableNow=True).foreachBatch(
    upsert_to_silver
).format("delta").option("checkpointLocation", silver_checkpoint_path).option(
    "mergeSchema", "true"
).outputMode(
    "update"
).trigger(
    once=True
).start().awaitTermination()

## ORDER SHIPMENTS

In [0]:
# Streaming read from bronze: order_shipments
df_bronze= spark.readStream\
.format('delta')\
.option("checkpointLocation", "/Volumes/shopvista/raw/checkpoints/bronze")\
.table(f'{catalog}.{bronze_schema}.order_shipments')

- Perform Transformations and Cleaning

In [0]:
# Convert order_dt to date type and inspect schema
df_bronze = df_bronze.withColumn('order_dt', F.to_date(F.col('order_dt')))
df_bronze.printSchema()

In [0]:
# change order_dt to datetime format
df_silver = df_bronze.withColumn(
    'order_dt',
    F.to_timestamp(F.col('order_dt'), 'yyyy-MM-dd')
)
#display(df_silver.limit(5))

In [0]:
#Convert `carrier` column to uppercase and trim whitespace.
df_silver = df_silver.withColumn(
    'carrier',
    F.upper(F.trim(F.col('carrier')))
)
#display(df_silver.limit(5))

In [0]:
# Add processed_time column
df_silver = df_silver.withColumn(
    'processed_time',
    F.current_timestamp()
)
#display(df_silver.limit(5))

## Save to Silver Table

In [0]:
# silver checkpoint location
silver_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/silver/fact_order_shipments"
print(silver_checkpoint_path)

In [0]:
# Upsert function: merge order_shipments microbatch into silver table
def upsert_to_silver(microBatchDF, batchId):
    table_name = f"{catalog}.silver.order_shipments"
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
        spark.sql(
            f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    else:
        deltaTable = DeltaTable.forName(spark, table_name)
        deltaTable.alias("silver_table").merge(
            microBatchDF.alias("batch_table"),
            "silver_table.order_shipments = batch_table.order_shipments AND silver_table.item_seq = batch_table.item_seq",
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()    

    



In [0]:
# This line is running a Structured Streaming job that:
# - Reads incremental data from Bronze (df).
# - For each batch → applies upsert_to_silver (update if exists, insert if not).
# - Writes into a Silver Delta table with schema evolution enabled.
# - Uses checkpointing for recovery.
# - Runs in batch-like mode (once or availableNow), not continuous streaming.

df_silver.writeStream.trigger(availableNow=True).foreachBatch(
    upsert_to_silver
).format("delta").option("checkpointLocation", silver_checkpoint_path).option(
    "mergeSchema", "true"
).outputMode(
    "update"
).trigger(
    once=True
).start().awaitTermination()